In [0]:
%run ./utils

# Imports

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.window import Window

# Load

In [0]:
dbutils.widgets.text("bronze_table_name", "tb_canais")
bronze_table_name = dbutils.widgets.get("bronze_table_name")

df_canais = (
    spark.table(f"{bronze_schema}.{bronze_table_name}")
        .distinct()
        .orderBy("id_canal")
)

display(df_canais.limit(5))

# Deduplicação

In [0]:
# Definição da janela agrupada pela chave primária e ordenada pelo timestamp decrescente
window_spec = Window.partitionBy("id_canal").orderBy(F.col("ingestion_datetime").desc())

df_canais = (
    df_canais
    .withColumn("rn", F.row_number().over(window_spec))
    .filter(F.col("rn") == 1)
    .drop("rn")
    .select(
        "id_canal",
        "nome_canal",
        "tipo_canal",
        "status_canal"
    )
)

# DF final deve ter 4 linhas
print(f"Total de linhas da tabela de canais: {df_canais.count()}")

display(df_canais)

# Transformação

In [0]:
df_cast = cast_id_to_string(df_canais)

# Salvamento

In [0]:
dbutils.widgets.text("silver_table_name", "tb_canais")
silver_table_name = dbutils.widgets.get("silver_table_name")

save_df(df_cast, silver_table_name)